# Notebook 3 - One-to-Many Transfer

This notebook asks whether a truth/correctness direction trained on one dataset transfers to other datasets. This is the core generalisation question.

The source dataset is `dbpedia_14`; the evaluation datasets include sentiment, QA, factual statements, TruthfulQA, and the local repeng truthful dataset.


In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "lie_detector_llm").exists():
            return candidate
    raise RuntimeError("Could not find the project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

RESULTS_DIR = PROJECT_ROOT / "results"
ACTIVATION_CACHE_DIR = PROJECT_ROOT / "data" / "activations"
RESULTS_DIR.mkdir(exist_ok=True)
ACTIVATION_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")


## Build the Evaluation Collection

`MAX_GROUPS` controls runtime and statistical stability. A larger value gives more reliable estimates, but activation extraction is slower. The saved project results were generated with `MAX_GROUPS = 50`.


In [ ]:
from lie_detector_llm.datasets import DEFAULT_DATASET_NAMES, build_dataset_collection

DATASET_NAMES = DEFAULT_DATASET_NAMES
MAX_GROUPS = 50

collection = build_dataset_collection(
    dataset_names=DATASET_NAMES,
    max_groups=MAX_GROUPS,
    seed=0,
)
dataset_names = collection.dataset_names()

display(collection.summary())
print("Evaluation datasets:", dataset_names)


## Transfer Configuration

The model is fixed to Phi-2 and the layer is set to 18, the best layer for the strongest transfer probes in our sweep. The experiment trains on the train split of `dbpedia_14` and evaluates on the test split of every dataset.


In [ ]:
from lie_detector_llm.experiment import DEFAULT_MODEL, PROBE_METHODS

MODEL_NAME = DEFAULT_MODEL
TRAIN_DATASET = "dbpedia_14"
LAYER_INDEX = 18
ACTIVATION_BATCH_SIZE = 2
MAX_LENGTH = 512
LOAD_IN_4BIT = False

print("Model:", MODEL_NAME)
print("Train dataset:", TRAIN_DATASET)
print("Layer:", LAYER_INDEX)
print("Probes:", PROBE_METHODS)


## Run Transfer for All Four Probes

The `is_in_distribution` row is the held-out test split of the training dataset. Every other row is out-of-distribution transfer.

Strong off-domain accuracy means that the direction learned from `dbpedia_14` is not only solving `dbpedia_14`; it is also useful for different kinds of correctness.


In [ ]:
import pandas as pd
from lie_detector_llm.experiment import run_transfer_experiment

rows = []
for method in PROBE_METHODS:
    out = run_transfer_experiment(
        collection=collection,
        train_dataset_name=TRAIN_DATASET,
        eval_dataset_names=dataset_names,
        model_name=MODEL_NAME,
        probe_method=method,
        layer_index=LAYER_INDEX,
        activation_batch_size=ACTIVATION_BATCH_SIZE,
        max_length=MAX_LENGTH,
        load_in_4bit=LOAD_IN_4BIT,
        show_progress=(method == PROBE_METHODS[0]),
        activation_cache_dir=ACTIVATION_CACHE_DIR,
    )
    rows.append(out.results)

transfer = pd.concat(rows, ignore_index=True)
transfer.to_csv(RESULTS_DIR / "phi2_one_to_many_transfer.csv", index=False)
display(transfer.sort_values(["probe_method", "eval_dataset"]))


## Plot Transfer Accuracy

The bars away from `dbpedia_14` are the important ones. They show whether a probe trained on topic classification transfers to other domains.


In [ ]:
import matplotlib.pyplot as plt
from lie_detector_llm.plotting import plot_transfer_results

fig, ax = plot_transfer_results(
    transfer,
    title=f"Phi-2 transfer from {TRAIN_DATASET}, layer {LAYER_INDEX}",
)
fig.savefig(RESULTS_DIR / "phi2_one_to_many_transfer.png", dpi=160, bbox_inches="tight")
plt.show()


## Summary Table

This table separates same-dataset performance from out-of-distribution performance. The transfer mean is the more important number for the research question.


In [ ]:
summary = (
    transfer.assign(eval_region=lambda df: df["is_in_distribution"].map({True: "in_distribution", False: "out_of_distribution"}))
    .groupby(["probe_method", "eval_region"], as_index=False)["grouped_accuracy"]
    .mean()
)
display(summary)


## Discussion

If a probe performs well only on `dbpedia_14`, it has learned a dataset-specific signal. If it also performs well on facts, sentiment, QA, and self-report truthfulness, that is evidence for a more general direction.

TruthfulQA should be interpreted separately. It is designed around common false beliefs and can be much harder than simple correctness datasets, especially for a smaller model like Phi-2.
